# SIRAP Backend · RT-DETR Ecuador

Active detector: **RT-DETRv2-S** with **native OpenVINO Runtime in Python**, using the Apache-2.0-licensed checkpoint `merve/license-plates-rtdetrv2`. New data and metrics are kept separate from historical YOLO results.


In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1" "gdown>=5,<6" "openvino>=2025.4,<2027"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="ecuador"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Profile:",COUNTRY,"| RT-DETRv2-S:",MODEL_ID)


## Profile-based inference
The backend loads the IR from `models/ecuador/detector/rtdetrv2_s/openvino/` if it exists; otherwise, it uses `models/shared/detector/rtdetrv2_s/openvino/`. In Colab, first run `python -m src.export_openvino --model-id merve/license-plates-rtdetrv2 --output-dir models/shared/detector/rtdetrv2_s/openvino` if the shared IR has not been exported yet.

In [ ]:
from pathlib import Path
from src.model import ALPRPipeline
from src.data_processing import load_image
SHARED_IR=REPO_ROOT/"models/shared/detector/rtdetrv2_s/openvino/openvino_model.xml"
if not SHARED_IR.is_file():
    from src.export_openvino import export_openvino_model
    export_openvino_model(MODEL_ID, SHARED_IR.parent)
pipeline=ALPRPipeline.from_config()
print(pipeline.detector.name)
IMAGE_PATH=None
if IMAGE_PATH: print(pipeline.recognize(load_image(Path(IMAGE_PATH).read_bytes())).as_dict())


## Demo video

Source video: [open in Google Drive](https://drive.google.com/file/d/1NWB5XcewwXKMdBpenRZA_-gmrwTqnVBm/view?usp=sharing). The cell downloads the file to Colab by its ID if it is not already present. It samples one frame every 2.5 seconds and saves predictions for review; evaluation requires temporal ground truth for this same video.

In [ ]:
from pathlib import Path
import cv2
import pandas as pd
import gdown

VIDEO_ID = "1NWB5XcewwXKMdBpenRZA_-gmrwTqnVBm"
VIDEO_PATH = Path(
    "/content/SIRAP_workspace/"
    "sirap_video_drive_1NWB5XcewwXKMdBpenRZA_-gmrwTqnVBm.mp4"
)
VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

if not VIDEO_PATH.is_file() or VIDEO_PATH.stat().st_size == 0:
    downloaded = gdown.download(
        id=VIDEO_ID,
        output=str(VIDEO_PATH),
        quiet=False,
    )
    if downloaded is None or not VIDEO_PATH.is_file() or VIDEO_PATH.stat().st_size == 0:
        raise RuntimeError(
            "The video could not be downloaded. Verify that Google Drive allows "
            "access through the shared link."
        )

OUT = "/content/decisions_rtdetr_ecuador.csv"
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open the downloaded video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
step = max(1, int(fps * 2.5))
rows = []
index = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    if index % step == 0:
        result = pipeline.recognize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        rows.append({
            "frame_index": index,
            "plate_pred": result.plate,
            "det_confidence": result.det_confidence,
            "ocr_confidence": result.ocr_confidence,
            "time_ms": result.timings_ms.get("total"),
        })
    index += 1

cap.release()
pd.DataFrame(rows).to_csv(OUT, index=False)
print("Video used:", VIDEO_PATH)
print("Saved", OUT, "frames analyzed", len(rows))